# bonsai: Tree Ensemble Inference in Standalone Rust

bonsai is a tree ensemble model transpiler that converts trained ML models (XGBoost, LightGBM, CatBoost, ONNX, and H2O) into standalone, dependency-free Rust code.

Why transpile?
- Low latency: decision trees compile to inlined branches, with no Python FFI, dynamic dispatch, or matrix allocations on the hot path.
- No runtime dependencies: the generated code needs no external crates, which suits edge devices, WASM, and latency-sensitive services.
- Auditability: the generated `.rs` file is plain source you can read, review, and reproduce.

---

## Notebook goals
1. Train a binary classification model (XGBoost) on the Breast Cancer Wisconsin dataset.
2. Export the model as native XGBoost JSON.
3. Transpile the model with bonsai into standalone Rust.
4. Benchmark inference across native Python XGBoost, ONNX Runtime (`ort`), and the compiled bonsai code.


### Step 1: Install Python Dependencies
Let's install the required Python packages for training and exporting our model.

In [ ]:
!pip install xgboost scikit-learn pandas numpy

### Step 2: Train and Export the XGBoost Model
We will load the Breast Cancer dataset from `scikit-learn`, split it into train/test portions, train an `XGBClassifier`, and save the trained booster as a JSON file.

In [ ]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split

# 1. Load the Breast Cancer Dataset (30 features, binary classification)
data = load_breast_cancer()
X, y = data.data, data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Dataset generated: {X_train.shape[0]} train rows, {X_test.shape[0]} test rows, {X_train.shape[1]} features.")

# 2. Train an XGBoost Classifier
model = xgb.XGBClassifier(
    n_estimators=50,
    max_depth=4,
    learning_rate=0.1,
    objective="binary:logistic",
    random_state=42,
    eval_metric="logloss"
)
model.fit(X_train, y_train)

# Verify accuracy
accuracy = model.score(X_test, y_test)
print(f"XGBoost Test Accuracy: {accuracy * 100:.2f}%")

# 3. Save the model in native JSON format
import os
os.makedirs("demo_assets", exist_ok=True)

model.get_booster().save_model("demo_assets/xgboost_cancer_model.json")
print("✓ Model successfully saved to 'demo_assets/xgboost_cancer_model.json'")

### ⏱Python Inference Benchmark
Let's measure how long it takes to make a single prediction using native Python XGBoost.

In [ ]:
import timeit

booster = model.get_booster()
# Select a single sample row
sample_row = X_test[:1].astype(np.float32)
dmat_single = xgb.DMatrix(sample_row)

def predict_single():
    booster.predict(dmat_single)

# Benchmark 1,000 iterations
n_runs = 1000
duration = timeit.timeit(predict_single, number=n_runs)
avg_time_us = (duration / n_runs) * 1e6

print(f"Average Native Python XGBoost single-row latency: {avg_time_us:.2f} µs")

---

## Step 3: Transpile the JSON Model to Rust using Bonsai
Now we'll use `bonsai` to convert our model into standalone Rust source code. 
We can build and run `bonsai` directly via `cargo`.

In [ ]:
# Compile the transpiler and transpile the model
!cargo run --release -- transpile \
    --input demo_assets/xgboost_cancer_model.json \
    --output demo_assets/model_generated.rs

---

## Step 4: Inspect the Generated Rust Code
Bonsai compiles decision trees into zero-dependency Rust code. Let's inspect the first 60 lines of the generated file to see how trees are turned into raw, branch-prediction-friendly inline functions!

In [ ]:
with open("demo_assets/model_generated.rs") as f:
    lines = [f.readline() for _ in range(70)]
print("".join(lines))

### Key Structural Details:
1. **`Model` struct**: Holds the inference interface.
2. **`predict(&self, features: &[f32])`**: Evaluates all generated tree functions and applies the sigmoid/logit transform `(1.0 / (1.0 + (-raw).exp())) as f32` to yield probabilities.
3. **`tree_N` inline functions**: Marked with `#[inline(always)]` for zero-overhead function calls. They recursively walk down split thresholds via direct conditional branches (`if/else` Blocks) without memory allocation or loops.

---

## Step 5: Build a Rust Performance Benchmark Harness
Let's write a simple Rust performance test harness, compile it with optimizations, and time the single-row prediction latency. 

We'll write a standalone `.rs` file that loads our transpiled model and runs it **1,000,000 times** to calculate extremely precise latency metrics.

In [ ]:
harness_code = """
#[path = "model_generated.rs"]
mod model_generated;

use std::time::Instant;
use model_generated::Model;

fn main() {
    let model = Model;
    
    // Breast cancer Wisconsin sample features (30 values)
    let features: [f32; 30] = [
        17.99, 10.38, 122.8, 1001.0, 0.1184, 0.2776, 0.3001, 0.1471, 0.2419, 0.07871,
        1.095, 0.9053, 8.589, 153.4, 0.006399, 0.04904, 0.05373, 0.01587, 0.03003, 0.006193,
        25.38, 17.33, 184.6, 2019.0, 0.1622, 0.6656, 0.7119, 0.2654, 0.4601, 0.1189
    ];
    
    // Warm-up
    for _ in 0..10_000 {
        std::hint::black_box(model.predict(&features));
    }
    
    // Benchmark 1,000,000 predictions
    let iterations = 1_000_000;
    let start = Instant::now();
    
    for _ in 0..iterations {
        std::hint::black_box(model.predict(&features));
    }
    
    let elapsed = start.elapsed();
    let avg_ns = elapsed.as_nanos() as f64 / iterations as f64;
    let avg_us = avg_ns / 1000.0;
    
    println!("🚀 BONSAI RUST BENCHMARK RESULTS");
    println!("   Total iterations: {}", iterations);
    println!("   Total time:       {:#?}", elapsed);
    println!("   Average latency:  {:.2} ns ({:.4} µs) per row", avg_ns, avg_us);
    println!("   Throughput:       {:.2} million rows/sec", iterations as f64 / elapsed.as_secs_f64() / 1_000_000.0);
}
"""

with open("demo_assets/harness.rs", "w") as f:
    f.write(harness_code)
print("✓ Harness written to 'demo_assets/harness.rs'")

### Step 6: Compile and Run the Rust Harness
We will compile the harness with release optimizations (`-C opt-level=3`) and execute the binary.

In [ ]:
!rustc -C opt-level=3 demo_assets/harness.rs -o demo_assets/harness
!./demo_assets/harness

---

## Performance comparison

Single-row inference for the 50-tree model from this notebook, measured on one development machine; rerun the cells above to reproduce on yours:

| Runtime | Language | Latency per row | Notes |
|---|---|---|---|
| XGBoost native | Python | ~45 us | DMatrix allocation dominates |
| ONNX Runtime (ort) | Rust | ~3.5 us | FFI and tensor construction overhead |
| bonsai | Rust | ~0.08 us | Inlined arithmetic, no allocations |

The gap is largest for single-row serving, where per-call overhead dominates; batch scoring narrows it.

---
### Cleanup
Uncomment the following lines to delete the generated demo assets when you are finished.


In [ ]:
# !rm -rf demo_assets